In [0]:
# ── CELL 1: Load Secrets & Configure ADLS ────────────────────
connection_string = dbutils.secrets.get(
    scope="clickstream-scope",
    key="eventhub-connection-string"
)
storage_account = dbutils.secrets.get(
    scope="clickstream-scope",
    key="storage-account-name"
)
storage_key = dbutils.secrets.get(
    scope="clickstream-scope",
    key="storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

EVENTHUB_NAME = "clickstream-events"
EVENTHUB_NAMESPACE = "evhbns-clickstream-dev"

print("Secrets loaded successfully.")

In [0]:
# ── CELL 2: Read Raw Stream from Event Hubs ──────────────────
kafka_options = {
    "kafka.bootstrap.servers": f"{EVENTHUB_NAMESPACE}.servicebus.windows.net:9093",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{connection_string}";',
    "subscribe": EVENTHUB_NAME,
    "startingOffsets": "latest",
    "kafka.request.timeout.ms": "60000",
    "kafka.session.timeout.ms": "30000",
    "failOnDataLoss": "false"
}

raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

print("Raw stream schema:")
raw_stream.printSchema()

In [0]:
# ── CELL 3: Parse & Enrich Stream ────────────────────────────
from pyspark.sql.functions import (
    col, from_json, to_timestamp, hour, dayofweek,
    when, lit, current_timestamp
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

clickstream_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("page_url", StringType(), True),
    StructField("referrer_url", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("browser", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("time_spent_seconds", IntegerType(), True)
])

parsed_stream = (
    raw_stream
    .select(
        from_json(
            col("value").cast("string"),
            clickstream_schema
        ).alias("data"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp")
    )
    .select("data.*", "partition", "offset", "kafka_timestamp")
)

enriched_stream = (
    parsed_stream
    .withColumn("event_timestamp", to_timestamp(col("timestamp")))
    .withColumn("hour_of_day", hour(col("event_timestamp")))
    .withColumn("day_of_week", dayofweek(col("event_timestamp")))
    .withColumn(
        "is_product_page",
        when(col("page_url").contains("/products"), lit(True))
        .otherwise(lit(False))
    )
    .withColumn(
        "is_conversion",
        when(col("event_type") == "purchase", lit(True))
        .otherwise(lit(False))
    )
    .withColumn("ingested_at", current_timestamp())
)

print("Enriched stream schema:")
enriched_stream.printSchema()

In [0]:
# ── CELL 4: Write Stream to ADLS Gen2 ────────────────────────
CONTAINER = "raw"
CHECKPOINT_PATH = (
    f"abfss://{CONTAINER}@{storage_account}"
    f".dfs.core.windows.net/checkpoints/clickstream"
)
OUTPUT_PATH = (
    f"abfss://{CONTAINER}@{storage_account}"
    f".dfs.core.windows.net/clickstream_events"
)

query = (
    enriched_stream
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("path", OUTPUT_PATH)
    .partitionBy("day_of_week", "hour_of_day")
    .trigger(processingTime="30 seconds")
    .start()
)

print(f"Streaming query started: {query.id}")
print(f"Writing to: {OUTPUT_PATH}")
print("Status:", query.status)

In [0]:
# DATABASE SCHEMA DEFINITION

# import psycopg2

# pg_host = dbutils.secrets.get(scope="clickstream-scope", key="postgres-host")
# pg_password = dbutils.secrets.get(scope="clickstream-scope", key="postgres-password")

# conn = psycopg2.connect(
#     host=pg_host,
#     database="clickstream_db",
#     user="pgadmin",
#     password=pg_password,
#     sslmode="require"
# )
# cur = conn.cursor()

# # Raw events table
# cur.execute("""
#     CREATE TABLE IF NOT EXISTS raw_events (
#         event_id VARCHAR(255) PRIMARY KEY,
#         user_id VARCHAR(255),
#         session_id VARCHAR(255),
#         event_timestamp TIMESTAMP,
#         page_url VARCHAR(500),
#         referrer_url VARCHAR(500),
#         event_type VARCHAR(50),
#         device_type VARCHAR(50),
#         browser VARCHAR(50),
#         country VARCHAR(100),
#         city VARCHAR(100),
#         product_id VARCHAR(255),
#         product_category VARCHAR(100),
#         time_spent_seconds INTEGER,
#         is_product_page BOOLEAN,
#         is_conversion BOOLEAN,
#         hour_of_day INTEGER,
#         day_of_week INTEGER,
#         ingested_at TIMESTAMP
#     );
# """)

# # Page traffic aggregates
# cur.execute("""
#     CREATE TABLE IF NOT EXISTS page_traffic (
#         id SERIAL PRIMARY KEY,
#         page_url VARCHAR(500),
#         event_type VARCHAR(50),
#         device_type VARCHAR(50),
#         event_count INTEGER,
#         avg_time_spent FLOAT,
#         window_start TIMESTAMP,
#         window_end TIMESTAMP,
#         created_at TIMESTAMP DEFAULT NOW()
#     );
# """)

# # Country traffic aggregates
# cur.execute("""
#     CREATE TABLE IF NOT EXISTS country_traffic (
#         id SERIAL PRIMARY KEY,
#         country VARCHAR(100),
#         city VARCHAR(100),
#         event_count INTEGER,
#         conversion_count INTEGER,
#         window_start TIMESTAMP,
#         window_end TIMESTAMP,
#         created_at TIMESTAMP DEFAULT NOW()
#     );
# """)

# # Device analytics aggregates
# cur.execute("""
#     CREATE TABLE IF NOT EXISTS device_analytics (
#         id SERIAL PRIMARY KEY,
#         device_type VARCHAR(50),
#         browser VARCHAR(50),
#         event_count INTEGER,
#         conversion_count INTEGER,
#         avg_time_spent FLOAT,
#         window_start TIMESTAMP,
#         window_end TIMESTAMP,
#         created_at TIMESTAMP DEFAULT NOW()
#     );
# """)

# conn.commit()
# cur.close()
# conn.close()
# print("All tables created successfully!")

All tables created successfully!


In [0]:
# ── CELL 5: Write Aggregates to PostgreSQL ───────────────────
from pyspark.sql.functions import window, count, avg, sum as spark_sum

pg_host = dbutils.secrets.get(scope="clickstream-scope", key="postgres-host")
pg_password = dbutils.secrets.get(scope="clickstream-scope", key="postgres-password")

JDBC_URL = f"jdbc:postgresql://{pg_host}:5432/clickstream_db?sslmode=require"
JDBC_PROPERTIES = {
    "user": "pgadmin",
    "password": pg_password,
    "driver": "org.postgresql.Driver"
}

def write_raw_events_to_pg(batch_df, batch_id):
    """Write raw events from each micro-batch to PostgreSQL."""
    if batch_df.count() == 0:
        return
    
    import psycopg2
    
    pg_host = dbutils.secrets.get(scope="clickstream-scope", key="postgres-host")
    pg_password = dbutils.secrets.get(scope="clickstream-scope", key="postgres-password")
    
    conn = psycopg2.connect(
        host=pg_host,
        database="clickstream_db",
        user="pgadmin",
        password=pg_password,
        sslmode="require"
    )
    cur = conn.cursor()
    
    rows = batch_df.select(
        "event_id", "user_id", "session_id",
        "event_timestamp", "page_url", "referrer_url",
        "event_type", "device_type", "browser",
        "country", "city", "product_id", "product_category",
        "time_spent_seconds", "is_product_page", "is_conversion",
        "hour_of_day", "day_of_week", "ingested_at"
    ).collect()
    
    insert_sql = """
        INSERT INTO raw_events (
            event_id, user_id, session_id, event_timestamp, page_url,
            referrer_url, event_type, device_type, browser, country, city,
            product_id, product_category, time_spent_seconds, is_product_page,
            is_conversion, hour_of_day, day_of_week, ingested_at
        ) VALUES (
            %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s, %s, %s, %s, %s
        )
        ON CONFLICT (event_id) DO NOTHING
    """
    
    for row in rows:
        cur.execute(insert_sql, (
            row.event_id, row.user_id, row.session_id,
            row.event_timestamp, row.page_url, row.referrer_url,
            row.event_type, row.device_type, row.browser,
            row.country, row.city, row.product_id, row.product_category,
            row.time_spent_seconds, row.is_product_page, row.is_conversion,
            row.hour_of_day, row.day_of_week, row.ingested_at
        ))
    
    conn.commit()
    cur.close()
    conn.close()
    print(f"Batch {batch_id}: wrote {len(rows)} raw events to PostgreSQL")

def write_page_traffic_to_pg(batch_df, batch_id):
    """Write page traffic aggregates to PostgreSQL."""
    if batch_df.count() == 0:
        return
    (
        batch_df
        .groupBy("page_url", "event_type", "device_type")
        .agg(
            count("*").alias("event_count"),
            avg("time_spent_seconds").alias("avg_time_spent")
        )
        .withColumn("window_start", current_timestamp())
        .withColumn("window_end", current_timestamp())
        .write
        .mode("append")
        .jdbc(JDBC_URL, "page_traffic", properties=JDBC_PROPERTIES)
    )
    print(f"Batch {batch_id}: wrote page traffic aggregates to PostgreSQL")

def write_country_traffic_to_pg(batch_df, batch_id):
    """Write country traffic aggregates to PostgreSQL."""
    if batch_df.count() == 0:
        return
    (
        batch_df
        .groupBy("country", "city")
        .agg(
            count("*").alias("event_count"),
            spark_sum(col("is_conversion").cast("integer")).alias("conversion_count")
        )
        .withColumn("window_start", current_timestamp())
        .withColumn("window_end", current_timestamp())
        .write
        .mode("append")
        .jdbc(JDBC_URL, "country_traffic", properties=JDBC_PROPERTIES)
    )
    print(f"Batch {batch_id}: wrote country traffic aggregates to PostgreSQL")

def write_device_analytics_to_pg(batch_df, batch_id):
    """Write device analytics aggregates to PostgreSQL."""
    if batch_df.count() == 0:
        return
    (
        batch_df
        .groupBy("device_type", "browser")
        .agg(
            count("*").alias("event_count"),
            spark_sum(col("is_conversion").cast("integer")).alias("conversion_count"),
            avg("time_spent_seconds").alias("avg_time_spent")
        )
        .withColumn("window_start", current_timestamp())
        .withColumn("window_end", current_timestamp())
        .write
        .mode("append")
        .jdbc(JDBC_URL, "device_analytics", properties=JDBC_PROPERTIES)
    )
    print(f"Batch {batch_id}: wrote device analytics to PostgreSQL")

# Start all PostgreSQL sinks using foreachBatch
raw_events_query = (
    enriched_stream
    .writeStream
    .foreachBatch(write_raw_events_to_pg)
    .option(
        "checkpointLocation",
        f"abfss://raw@{storage_account}.dfs.core.windows.net/checkpoints/raw_events_pg"
    )
    .trigger(processingTime="30 seconds")
    .start()
)

page_traffic_query = (
    enriched_stream
    .writeStream
    .foreachBatch(write_page_traffic_to_pg)
    .option(
        "checkpointLocation",
        f"abfss://raw@{storage_account}.dfs.core.windows.net/checkpoints/page_traffic_pg"
    )
    .trigger(processingTime="30 seconds")
    .start()
)

country_traffic_query = (
    enriched_stream
    .writeStream
    .foreachBatch(write_country_traffic_to_pg)
    .option(
        "checkpointLocation",
        f"abfss://raw@{storage_account}.dfs.core.windows.net/checkpoints/country_traffic_pg"
    )
    .trigger(processingTime="30 seconds")
    .start()
)

device_analytics_query = (
    enriched_stream
    .writeStream
    .foreachBatch(write_device_analytics_to_pg)
    .option(
        "checkpointLocation",
        f"abfss://raw@{storage_account}.dfs.core.windows.net/checkpoints/device_analytics_pg"
    )
    .trigger(processingTime="30 seconds")
    .start()
)

print("All PostgreSQL streaming queries started!")
print("raw_events_query:", raw_events_query.status)
print("page_traffic_query:", page_traffic_query.status)
print("country_traffic_query:", country_traffic_query.status)
print("device_analytics_query:", device_analytics_query.status)

All PostgreSQL streaming queries started!
raw_events_query: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
page_traffic_query: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
country_traffic_query: {'message': 'Writing offsets to log', 'isDataAvailable': False, 'isTriggerActive': True}
device_analytics_query: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [0]:
# ── CELL 6: Verify PostgreSQL Data ───────────────────────────
import psycopg2

conn = psycopg2.connect(
    host=pg_host,
    database="clickstream_db",
    user="pgadmin",
    password=pg_password,
    sslmode="require"
)
cur = conn.cursor()

for table in ["raw_events", "page_traffic", "country_traffic", "device_analytics"]:
    cur.execute(f"SELECT COUNT(*) FROM {table};")
    count = cur.fetchone()[0]
    print(f"{table}: {count} rows")

cur.close()
conn.close()

raw_events: 4809 rows
page_traffic: 3113 rows
country_traffic: 4803 rows
device_analytics: 373 rows
